In [18]:
# Messy Notes → Clean Ledger (FREE, Colab, T4)

!pip -q install transformers accelerate bitsandbytes sentencepiece gradio pydantic json-repair

# Loading model (Qwen) with proper 4-bit config
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

def llm_generate(prompt: str, max_new_tokens: int = 700) -> str:
    """
    Qwen: build chat with apply_chat_template, then decode ONLY generated tokens.
    This prevents 'system/user' text from appearing in the output.
    """
    messages = [
        {"role": "system", "content": "You are a careful assistant that follows instructions exactly."},
        {"role": "user", "content": prompt},
    ]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    # Only decode the generated continuation
    gen_ids = output_ids[0][input_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# Schema (Pydantic)
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Dict

class Split(BaseModel):
    type: Literal["none", "equal", "custom"] = "none"
    details: Optional[Dict[str, float]] = None

class Item(BaseModel):
    merchant: str
    amount: float
    currency: str = "CAD"
    category: Literal["Transport", "Food", "Rent", "Groceries", "Bills", "Shopping", "Other"] = "Other"
    date: str
    people: List[str] = Field(default_factory=lambda: ["me"])
    split: Split = Field(default_factory=Split)
    confidence: float = 0.5
    assumptions: List[str] = Field(default_factory=list)

class ParseResult(BaseModel):
    items: List[Item] = Field(default_factory=list)
    global_assumptions: List[str] = Field(default_factory=list)
    warnings: List[str] = Field(default_factory=list)


# Robust JSON parsing + repair
import json
from json_repair import repair_json

TODAY = "2025-12-27"
ALLOWED_CATEGORIES = ["Transport","Food","Rent","Groceries","Bills","Shopping","Other"]
ALLOWED_SPLITS = ["none","equal","custom"]

SCHEMA_HINT = f"""
You MUST output ONLY valid JSON. No markdown. No code fences. No extra text.

Return JSON with this exact shape:
{{
  "items":[
    {{
      "merchant":"string",
      "amount": number,
      "currency":"CAD",
      "category": one of {ALLOWED_CATEGORIES},
      "date":"YYYY-MM-DD",
      "people":["me","optional_other_names"],
      "split":{{"type": one of {ALLOWED_SPLITS}, "details": object_or_null }},
      "confidence": number_between_0_and_1,
      "assumptions":["strings"]
    }}
  ],
  "global_assumptions":["strings"],
  "warnings":["strings"]
}}

Rules:
- Choose EXACTLY ONE category from {ALLOWED_CATEGORIES}.
- Choose EXACTLY ONE split.type from {ALLOWED_SPLITS}.
- Do NOT invent amounts or merchants.
- If unclear, LOWER confidence and add warnings/assumptions.
- If no date is given, use TODAY={TODAY}.
- If currency is not given, default to CAD.
- If someone "owes" someone, use split.type="custom" and details like {{"sarah": 12.0}}.
"""

def extract_first_json_object(text: str) -> str:
    start = text.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError("Incomplete JSON object in model output.")

def safe_json_loads(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        fixed = repair_json(text)
        return json.loads(fixed)

def _clean_choice(value: str, allowed: list[str], default: str) -> str:
    if not isinstance(value, str):
        return default
    tokens = [t.strip() for t in value.replace("/", "|").split("|")]
    for t in tokens:
        if t in allowed:
            return t
    lower_map = {a.lower(): a for a in allowed}
    for t in tokens:
        if t.lower() in lower_map:
            return lower_map[t.lower()]
    return default

def coerce_model_output(obj: dict) -> dict:
    if "items" not in obj or not isinstance(obj["items"], list):
        obj["items"] = []

    for it in obj["items"]:
        if not isinstance(it, dict):
            continue

        it["category"] = _clean_choice(it.get("category", ""), ALLOWED_CATEGORIES, "Other")

        split = it.get("split", {})
        if not isinstance(split, dict):
            split = {}
        split["type"] = _clean_choice(split.get("type", ""), ALLOWED_SPLITS, "none")
        if split["type"] != "custom":
            split["details"] = None
        it["split"] = split

        try:
            c = float(it.get("confidence", 0.5))
        except:
            c = 0.5
        it["confidence"] = max(0.0, min(1.0, c))

        if not it.get("date"):
            it["date"] = TODAY
        if not it.get("currency"):
            it["currency"] = "CAD"
        if "people" not in it or not isinstance(it["people"], list) or len(it["people"]) == 0:
            it["people"] = ["me"]
        if "assumptions" not in it or not isinstance(it["assumptions"], list):
            it["assumptions"] = []

    if "global_assumptions" not in obj or not isinstance(obj["global_assumptions"], list):
        obj["global_assumptions"] = []
    if "warnings" not in obj or not isinstance(obj["warnings"], list):
        obj["warnings"] = []
    return obj


def parse_notes(notes: str, debug: bool = False) -> ParseResult:
    notes = (notes or "").strip()
    if not notes:
        return ParseResult(items=[], global_assumptions=[], warnings=["Empty input."])

    prompt = f"""You MUST return exactly ONE JSON object and NOTHING else.
End your output with <END_JSON> and do not write anything after it.

TODAY={TODAY}

{SCHEMA_HINT}

TEXT:
{notes}

<END_JSON>
"""
    raw = llm_generate(prompt, max_new_tokens=900)
    raw = raw.split("<END_JSON>")[0].strip()

    if debug:
        print("---- RAW MODEL OUTPUT (first 600 chars) ----")
        print(raw[:600])
        print("------------------------------------------")

    try:
        jtext = extract_first_json_object(raw)
        jobj = safe_json_loads(jtext)
        jobj = coerce_model_output(jobj)
        return ParseResult(**jobj)

    except Exception as e1:
        # If first attempt didn't even include '{', do a repair prompt that explicitly starts with '{'
        repair_prompt = f"""Return ONLY corrected JSON and end with <END_JSON>.
IMPORTANT: Your output MUST start with '{{' as the first character.

TODAY={TODAY}

SCHEMA:
{SCHEMA_HINT}

ORIGINAL_TEXT:
{notes}

BROKEN_OUTPUT:
{raw}

ERROR:
{str(e1)}

<END_JSON>
"""
        raw2 = llm_generate(repair_prompt, max_new_tokens=900)
        raw2 = raw2.split("<END_JSON>")[0].strip()

        if debug:
            print("---- RAW REPAIR OUTPUT (first 600 chars) ----")
            print(raw2[:600])
            print("--------------------------------------------")

        jtext2 = extract_first_json_object(raw2)
        jobj2 = safe_json_loads(jtext2)
        jobj2 = coerce_model_output(jobj2)
        return ParseResult(**jobj2)


test_notes = "uber 18, latte 6.5, split dinner 64 with 3 ppl i owe sarah 12, rent 1200"
res = parse_notes(test_notes, debug=True)
print(res.model_dump_json(indent=2))


# Gradio app
import gradio as gr
import pandas as pd
import tempfile, os

def result_to_df(res: ParseResult) -> pd.DataFrame:
    rows = []
    for it in res.items:
        rows.append({
            "date": it.date,
            "merchant": it.merchant,
            "amount": it.amount,
            "currency": it.currency,
            "category": it.category,
            "people": ", ".join(it.people),
            "split_type": it.split.type,
            "split_details": "" if not it.split.details else str(it.split.details),
            "confidence": float(it.confidence),
            "assumptions": " | ".join(it.assumptions) if it.assumptions else ""
        })
    return pd.DataFrame(rows)

def parse_and_render(notes: str):
    if not notes or not notes.strip():
        return pd.DataFrame([]), "{}", None, "Please paste some notes."

    try:
        res = parse_notes(notes.strip(), debug=False)
        df = result_to_df(res)

        tmpdir = tempfile.mkdtemp()
        csv_path = os.path.join(tmpdir, "ledger.csv")
        df.to_csv(csv_path, index=False)

        json_text = res.model_dump_json(indent=2)
        status = f"Parsed {len(df)} item(s)."
        return df, json_text, csv_path, status

    except Exception as e:
        return pd.DataFrame([]), "{}", None, f"Error: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("# 🧾 Messy Notes → Clean Ledger (Free GenAI in Colab)")
    gr.Markdown("Paste messy spending notes → get a clean ledger table + assumptions + warnings + confidence.")

    notes = gr.Textbox(
        lines=7,
        label="Paste your messy notes",
        placeholder="e.g., uber 18, latte 6.5, split dinner 64 with 3 ppl i owe sarah 12, rent 1200"
    )

    btn = gr.Button("Parse ✨")
    status = gr.Textbox(label="Status", interactive=False)

    out_df = gr.Dataframe(label="Ledger Table", wrap=True)
    out_json = gr.Code(label="Integrity Panel (JSON)", language="json")
    download = gr.File(label="Download CSV")

    btn.click(parse_and_render, inputs=[notes], outputs=[out_df, out_json, download, status])

demo.launch(share=True)

---- RAW MODEL OUTPUT (first 600 chars) ----
```json
{
  "items": [
    {
      "merchant": "Uber",
      "amount": 18.0,
      "currency": "USD",
      "category": "Transport",
      "date": "2025-12-27",
      "people": ["me", "Sarah"],
      "split": {
        "type": "custom",
        "details": {"Sarah": 12.0}
      },
      "confidence": 0.9,
      "assumptions": ["No assumptions needed for this transaction."]
    },
    {
      "merchant": "Latte",
      "amount": 6.5,
      "currency": "USD",
      "category": "Food",
      "date": "2025-12-27",
      "people": ["me"],
      "split": {
        "type": "none"
      },
      "confi
------------------------------------------
{
  "items": [
    {
      "merchant": "Uber",
      "amount": 18.0,
      "currency": "USD",
      "category": "Transport",
      "date": "2025-12-27",
      "people": [
        "me",
        "Sarah"
      ],
      "split": {
        "type": "custom",
        "details": {
          "Sarah": 12.0
        }
   